In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder


In [2]:
df = pd.read_csv('diabetic_data.csv')
df_map = pd.read_csv('IDS_mapping.csv')

disposition = df_map.to_numpy()
disposition = disposition[9:40]
disposition = pd.DataFrame(disposition[1:], columns=[disposition[0,0], disposition[0,1]])
disposition['Expired'] = disposition['description'].str.contains('Expired|Hospice').astype(int)
expired_disposition_id = disposition[disposition['Expired'] == 1]['discharge_disposition_id'].tolist()
print(expired_disposition_id)

['11', '13', '14', '19', '20', '21']


We remove all instances of patients who expired since they can not be target.

In [3]:
df = df[~df['discharge_disposition_id'].isin(expired_disposition_id)]


In [4]:
df.isnull().sum()

encounter_id                    0
patient_nbr                     0
race                            0
gender                          0
age                             0
weight                          0
admission_type_id               0
discharge_disposition_id        0
admission_source_id             0
time_in_hospital                0
payer_code                      0
medical_specialty               0
num_lab_procedures              0
num_procedures                  0
num_medications                 0
number_outpatient               0
number_emergency                0
number_inpatient                0
diag_1                          0
diag_2                          0
diag_3                          0
number_diagnoses                0
max_glu_serum               96420
A1Cresult                   84748
metformin                       0
repaglinide                     0
nateglinide                     0
chlorpropamide                  0
glimepiride                     0
acetohexamide 

We are dropping colums with high null value counts like `max_glu_serum`, `A1Cresult`

In [5]:
df = df.drop(columns= ['max_glu_serum', 'A1Cresult'])

**Simplify:** Convert the `target` column into a binary value: 1 (target within 30 days) and 0 (No readmission or $>30$ days).

In [6]:
df = df.rename(columns={'readmitted': 'target'})
df['target'].value_counts()

target
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64

In [7]:
df['target'] = df['target'].apply(lambda x: 1 if x == '<30' else 0)

In [8]:
df['target'].value_counts()

target
0    90409
1    11357
Name: count, dtype: int64

In [9]:
df['weight'].value_counts()
df['payer_code'].value_counts()
df['medical_specialty'].value_counts()

medical_specialty
?                         49949
InternalMedicine          14635
Emergency/Trauma           7565
Family/GeneralPractice     7440
Cardiology                 5352
                          ...  
Dermatology                   1
SportsMedicine                1
Speech                        1
Perinatology                  1
Neurophysiology               1
Name: count, Length: 73, dtype: int64

In [10]:
df = df.drop(columns=['weight', 'payer_code', 'medical_specialty'])

Here we conduct an experiment to find the top 15 most important features

In [11]:
df.dropna(inplace=True)

In [12]:
X = df.drop(columns=['target'])
y = df['target']

# Encode categorical columns
categorical_cols = X.select_dtypes(include=['object']).columns
for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X, y)

importances = model.feature_importances_
feature_importance = pd.DataFrame({
    'feature': X.columns, 
    'importance': importances
    }).sort_values(by='importance', ascending=False)
top_features = feature_importance.iloc[:15]
print(top_features)

C:\Users\arewa\AppData\Local\Temp\ipykernel_12964\2638210176.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=['object']).columns


                     feature  importance
0               encounter_id    0.094677
1                patient_nbr    0.093446
9         num_lab_procedures    0.078041
15                    diag_1    0.077899
16                    diag_2    0.076292
17                    diag_3    0.075545
11           num_medications    0.067481
8           time_in_hospital    0.046797
14          number_inpatient    0.040592
4                        age    0.039987
10            num_procedures    0.032886
6   discharge_disposition_id    0.032333
18          number_diagnoses    0.029573
36                   insulin    0.023835
5          admission_type_id    0.022908


Now our dataframe contains only the most important features

In [13]:
df= df[top_features['feature'].to_list()]
df.head()

,encounter_id,patient_nbr,num_lab_procedures,diag_1,diag_2,diag_3,num_medications,time_in_hospital,number_inpatient,age,num_procedures,discharge_disposition_id,number_diagnoses,insulin,admission_type_id
0,2278392,8222157,41,250.83,?,?,1,1,0,[0-10),0,25,1,No,6
1,149190,55629189,59,276,250.01,255,18,3,0,[10-20),0,1,9,Up,1
2,64410,86047875,11,648,250,V27,13,2,1,[20-30),5,1,6,No,1
3,500364,82442376,44,8,250.43,403,16,2,0,[30-40),1,1,7,Up,1
4,16680,42519267,51,197,157,250,8,1,0,[40-50),0,1,5,Steady,1


In [15]:
X = df

# Encode categorical columns
categorical_cols = X.select_dtypes(include=['object']).columns
for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])

# Build the Model
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
model.fit(X_train, y_train)

# Generate Risk Scores for the ENTIRE Dataset
# We want a probability score (0.0 to 1.0) for every row for our dashboard
df['Readmission_Probability'] = model.predict_proba(X)[:, 1]

# Create Risk Tiers for easy filtering in Tableau
def categorize_risk(prob):
    if prob > 0.7: return 'High Risk'
    elif prob > 0.3: return 'Medium Risk'
    else: return 'Low Risk'

df['Risk_Tier'] = df['Readmission_Probability'].apply(categorize_risk)

# 7. Export for Tableau Public
# We only export the columns we need to keep the file size small for the web
# Include the target variable for analysis in Tableau
output_cols = top_features['feature'].tolist()
output_cols.extend(['Risk_Tier', 'Readmission_Probability', 'target'])

# Ensure target is present after feature selection
if 'target' not in df.columns:
    df['target'] = y.values

df[output_cols].to_csv('Healthcare_AI_Project_Data.csv', index=False)
print("Success: 'Healthcare_AI_Project_Data.csv' is ready for Tableau!")

C:\Users\arewa\AppData\Local\Temp\ipykernel_12964\354121129.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=['object']).columns


Success: 'Healthcare_AI_Project_Data.csv' is ready for Tableau!
